In [1]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

sys.path.insert(0, str(Path.cwd().parent))
from src.config import RAW_FILES, REVIEW_SAMPLE_JSONL, CLEAN_PARQUET_PATH
from src.cleaning import run_cleaning_pipeline

# 1. Загружаем отладочный срез и справочник заведений
df_reviews = pd.read_json(REVIEW_SAMPLE_JSONL, lines=True)
df_business = pd.read_json(RAW_FILES["business"], lines=True)

# 2. Запускаем общие функции очистки из модуля src/cleaning.py
df_clean_nb, summary_df = run_cleaning_pipeline(df_reviews, df_business)

display(Markdown("### Итоговая сводка «Было — Стало» по этапам очистки (Шаг 12)"))
display(summary_df)

total_lost = len(df_reviews) - len(df_clean_nb)
print(f"ИТОГО: Было {len(df_reviews):,} строк -> Стало {len(df_clean_nb):,} строк (удалено {total_lost} строк, или {total_lost/len(df_reviews)*100:.3f}%)")
display(df_clean_nb.head(3))

### Итоговая сводка «Было — Стало» по этапам очистки (Шаг 12)

,Этап очистки,Строк до,Строк после,Удалено строк,"Доля потерь, %"
0,1. Соединение таблиц (JOIN),200000,200000,0,0.000
1,2. Нормализация имён столбцов,200000,200000,0,0.000
2,3. Обработка скрытых и явных пропусков,200000,199990,10,0.005
3,4. Приведение типов и дат,199990,199990,0,0.000
4,5. Удаление дубликатов,199990,199990,0,0.000
5,6. Нормализация категорий и городов,199990,199990,0,0.000
6,7. Очистка текстового поля text,199990,199980,10,0.005
7,8. Проверка предметных ограничений,199980,199980,0,0.000
8,9. Обработка выбросов (IQR vs 3-sigma),199980,199980,0,0.000


ИТОГО: Было 200,000 строк -> Стало 199,980 строк (удалено 20 строк, или 0.010%)


,review_id,user_id,business_id,stars,useful,funny,cool,text,date,business_name,...,state,postal_code,latitude,longitude,business_stars,review_count,is_open,main_category,stars_100,is_useful_outlier
0,--0HN389rkL6zLzqUrpGog,wjtj6GJ55PlAFo-bZUanSQ,iDtIMWJVIGYspg5JvJKaOw,2,2,0,0,I've been curious to try Mad Mex since it open...,2014-08-26 14:05:06,Mad Mex - Wynnewood,...,PA,19096,40.001846,-75.271637,2.5,248,False,Food,40.0,False
1,--2z5bwg3PPrWnr9wI84cA,J8x_xZIrxJd0aqG8ly0IAQ,D3DMIr-I0AKzlIKdErxHGg,5,2,0,2,This place is located where Salt used to be. T...,2014-03-03 18:20:19,Nathalie's,...,MO,63108,38.641399,-90.252411,3.0,67,False,American (Traditional),100.0,False
2,--46suwqJVAnei8ng2TeDw,yRgmyex9EXjFN5zDbxQkfw,qlC5ojQRLe26shP1Kt7hQg,3,0,0,0,I'm giving this Waffle house 3 stars because o...,2012-09-13 00:46:06,Waffle House,...,LA,70094,29.909586,-90.206871,2.5,27,True,Diners,60.0,False


In [2]:
import subprocess

# 1. Запускаем сквозной сценарий src.pipeline из командной строки
res = subprocess.run(
    [sys.executable, "-m", "src.pipeline", "--sample"],
    cwd=str(Path.cwd().parent),
    capture_output=True,
    text=True,
    encoding="utf-8"
)
print("Журнал выполнения консольного модуля src.pipeline:")
print(res.stdout)

# 2. Читаем файл clean.parquet, созданный модулем pipeline.py, и сверяем с результатом блокнота
df_clean_cli = pd.read_parquet(CLEAN_PARQUET_PATH)
pd.testing.assert_frame_equal(df_clean_nb, df_clean_cli, check_categorical=False, check_dtype=False)
print("✅ Проверка согласованности пройдена: наборы из блокнота и из CLI совпадают построчно на 100%!")

# 3. Проверка идемпотентности (повторный запуск)
subprocess.run([sys.executable, "-m", "src.pipeline", "--sample"], cwd=str(Path.cwd().parent))
df_clean_cli_second = pd.read_parquet(CLEAN_PARQUET_PATH)
pd.testing.assert_frame_equal(df_clean_cli, df_clean_cli_second)
print("✅ Проверка идемпотентности пройдена: повторный запуск даёт идентичный результат!")

Журнал выполнения консольного модуля src.pipeline:
2026-10-30 19:54:22,228 INFO Загрузка входных данных: D:\Labs\PythonLab1\data\interim\review_sample_200k.jsonl
2026-10-30 19:54:26,240 INFO merge_sources: строк до=200000, после=200000, несопоставившихся записей=0
2026-10-30 19:54:26,246 INFO normalize_column_names: столбцов=19
2026-10-30 19:54:27,345 INFO handle_hidden_missing: было=200000, стало=199990 (удалено=10)
2026-10-30 19:54:27,411 INFO cast_types_and_dates: было=199990, стало=199990
2026-10-30 19:54:28,100 INFO remove_duplicates: было=199990, стало=199990 (удалено=0)
2026-10-30 19:54:28,358 INFO normalize_categories: уникальных городов=320, категорий=260
2026-10-30 19:54:31,932 INFO clean_text_fields: было=199990, стало=199980 (удалено=10)
2026-10-30 19:54:31,937 INFO validate_domain_constraints: было=199980, стало=199980
2026-10-30 19:54:31,948 INFO detect_and_handle_outliers: выбросов по IQR=20680 (>2.5), по 3-сигма=2492 (>7.1), порог усечения 99.9%=19
2026-10-30 19:54:32,5